In [1]:
import ollama
import json
import lancedb
from lancedb.rerankers import ColbertReranker
from tqdm.notebook import tqdm
from pydantic import BaseModel, Field
from typing import Literal
from devtools import debug
import instructor

METRICS_FIGURE_NAME = "Contextual retrieval performance metrics (document slice only)"


OWN_TABLE_NAME = "my_anthropic_sliding_table"
BASELINE_TABLE_NAME = "anthropic_control_table"
# BASELINE_TABLE_NAME = "semantic_table"
QNA_FILE_PATH = "q_and_a/Gemini/scientific_multi_chunk_control.json"
MODEL_TAG = "qwen3-vl:8b-instruct-q4_K_M"

client = instructor.from_provider(
	f"ollama/{MODEL_TAG}",
	base_url="http://localhost:11434/v1",
	mode=instructor.Mode.JSON,
)

class RelevanceEvaluation(BaseModel):
	chain_of_thought: str = Field(
		..., 
		description="A brief reasoning step explaining why the score was given."
	)
	score: Literal[0,1,2,3] = Field(
		..., 
		description="The relevance score (0, 1, 2, or 3) based on the grading rubric."
	)

def grade_chunk_relevance(question: str, chunk_text: str, model_name: str) -> RelevanceEvaluation:
	"""
	Uses Qwen to grade a single chunk against a question.
	Returns the integer score (0-3).
	"""
	
	# Precise rubric for the system prompt
	system_prompt = """
	You are an impartial expert judge evaluating retrieval quality for a RAG system.
	Evaluate the relevance of the PASSAGE to the QUESTION using this strict scale:
	
	0: Irrelevant. The passage is on a different topic or does not help.
	1: Tangential. Mentions related entities but does not elaborate or provides explicit answer to the question.
	2: Relevant/Partial. Provides useful context or a partial answer.
	3: Highly Relevant. Contains the direct answer or core evidence required.
	"""

	try:
		resp = client.create(
			model=model_name,
			response_model=RelevanceEvaluation,
			messages=[
				{"role": "system", "content": system_prompt},
				{"role": "user", "content": f"QUESTION: {question}\nPASSAGE: {chunk_text}"}
			],
			temperature=0
		)
		return resp
	except Exception as e:
		print(f"Error grading chunk: {e}")
		return 0 # Fail-safe: assume irrelevant if model crashes

/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in ColPaliEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/home/martin/projects/TDK/Document_Slice_Contextual_Retrieval/.venv/lib/python3.13/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in SigLipEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [2]:
db = lancedb.connect("./db")
own_table = db.open_table(OWN_TABLE_NAME)
baseline_table = db.open_table(BASELINE_TABLE_NAME)
reranker = ColbertReranker()

with open(QNA_FILE_PATH, "r") as f:
	qna = json.load(f)

Loading ColBERTRanker model colbert-ir/colbertv2.0 (this message can be suppressed by setting verbose=0)
No device set
Using device cuda
No dtype set
Using dtype torch.float32
Loading model colbert-ir/colbertv2.0, this might take a while...
Linear Dim set to: 128 for downcasting


In [3]:
qna[0]['question']

'How did the distinct components of the Gander project inform the structure of the derived conceptual framework for open science in software engineering?'

In [4]:
import numpy as np

own_metrics_per_question = []
baseline_metrics_per_question = []

own_avg_recall_at_n = {}
baseline_avg_recall_at_n = {}
own_mrr_at_n = {}
baseline_mrr_at_n = {}
own_mar_at_n = {}
baseline_mar_at_n = {}

recall_diffs_at_n = {5: [], 10: [], 15: [], 20: []}

for question in tqdm(qna, desc="Processing questions..."):
	question_prompt = question["question"]
	supporting_chunk_ids = set(question["supporting_chunks"])

	own_df = own_table.search(question_prompt, query_type="hybrid", vector_column_name="vector", fts_columns="text") \
			.rerank(reranker=reranker) \
			.limit(20) \
			.to_pandas()
	
	baseline_df = baseline_table.search(question_prompt, query_type="hybrid", vector_column_name="vector", fts_columns="text") \
			.rerank(reranker=reranker) \
			.limit(20) \
			.to_pandas()

	own_metrics = {}
	baseline_metrics = {}

	for i in range(5,21,5):
		own_df_cutoff = own_df.head(i)
		baseline_df_cutoff = baseline_df.head(i)

		own_retreived_chunk_ids = own_df_cutoff['id'].tolist()
		own_recall_at_i = sum([1 if id in supporting_chunk_ids else 0 for id in own_retreived_chunk_ids]) / len(supporting_chunk_ids)

		baseline_retreived_chunk_ids = baseline_df_cutoff['id'].tolist()
		baseline_recall_at_i = sum([1 if id in supporting_chunk_ids else 0 for id in baseline_retreived_chunk_ids]) / len(supporting_chunk_ids)

		own_metrics[i] = {'recall': 0.0, 'reciprocal_rank': np.nan}
		baseline_metrics[i] = {'recall': 0.0, 'reciprocal_rank': np.nan}
		own_metrics[i]['recall'] = own_recall_at_i
		baseline_metrics[i]['recall'] = baseline_recall_at_i
		recall_diffs_at_n[i].append(own_recall_at_i - baseline_recall_at_i)

		first_relevant_found = False
		relevant_chunk_ranks = []
		for rank, chunk in enumerate(own_retreived_chunk_ids):
			if chunk in supporting_chunk_ids:
				if not first_relevant_found:
					own_metrics[i]['reciprocal_rank'] = 1.0 / (rank + 1)
					first_relevant_found = True
				relevant_chunk_ranks.append(rank + 1)
		
		own_metrics[i]['average_rank'] = sum(relevant_chunk_ranks) / len(relevant_chunk_ranks) if len(relevant_chunk_ranks) > 0 else np.nan

		first_relevant_found = False
		relevant_chunk_ranks = []
		for rank, chunk in enumerate(baseline_retreived_chunk_ids):
			if chunk in supporting_chunk_ids:
				if not first_relevant_found:
					baseline_metrics[i]['reciprocal_rank'] = 1.0 / (rank + 1)
					first_relevant_found = True
				relevant_chunk_ranks.append(rank + 1)
		
		baseline_metrics[i]['average_rank'] = sum(relevant_chunk_ranks) / len(relevant_chunk_ranks) if len(relevant_chunk_ranks) > 0 else np.nan

	own_metrics_per_question.append(own_metrics)
	baseline_metrics_per_question.append(baseline_metrics)

Processing questions...:   0%|          | 0/250 [00:00<?, ?it/s]

<All keys matched successfully>
<All keys matched successfully>


In [5]:
largest_diff = max(recall_diffs_at_n[5])
questions_with_largest_diff = [qna[i]['question'] for i, diff in enumerate(recall_diffs_at_n[5]) if diff > 0.5]
print(f"Questions with largest recall difference at 5: {questions_with_largest_diff}")
with open("largest_diff_questions.txt", "w") as f:
	f.write("\n".join(questions_with_largest_diff))
largest_diff_index = recall_diffs_at_n[5].index(largest_diff)
question_with_largest_diff = qna[largest_diff_index]['question']
print(f"Largest recall difference at 5: {largest_diff}")
print(f"Question with largest recall difference at 5: {question_with_largest_diff}")

Questions with largest recall difference at 5: ["How does the study characterize the trade-off between short-term and long-term resilience regarding the operation of diesel generators in the St. Mary's microgrid?"]
Largest recall difference at 5: 0.6666666666666666
Question with largest recall difference at 5: How does the study characterize the trade-off between short-term and long-term resilience regarding the operation of diesel generators in the St. Mary's microgrid?


In [6]:
sorted(recall_diffs_at_n[5], reverse=True)

[0.6666666666666666,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.5,
 0.33333333333333337,
 0.3333333333333333,
 0.25,
 0.25,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0,
 0.0

In [7]:

for i in range(5, 21, 5):
	none_counter = 0
	mrr_counter = 0
	for question in own_metrics_per_question:
		if question[i]['average_rank'] is np.nan:
			none_counter += 1
		if question[i]['reciprocal_rank'] is np.nan:
			mrr_counter += 1
	print(f"--- N={i} ---")
	print(f"Number of N values with NaN AR in own metrics: {none_counter}/{len(own_metrics_per_question)}")
	print(f"Number of N values with NaN MRR in own metrics: {mrr_counter}/{len(own_metrics_per_question)}")

--- N=5 ---
Number of N values with NaN AR in own metrics: 21/250
Number of N values with NaN MRR in own metrics: 21/250
--- N=10 ---
Number of N values with NaN AR in own metrics: 9/250
Number of N values with NaN MRR in own metrics: 9/250
--- N=15 ---
Number of N values with NaN AR in own metrics: 6/250
Number of N values with NaN MRR in own metrics: 6/250
--- N=20 ---
Number of N values with NaN AR in own metrics: 6/250
Number of N values with NaN MRR in own metrics: 6/250


In [8]:
import numpy as np

for i in range(5,21,5):
    own_avg_recall_at_n[i] = np.average([metrics[i]['recall'] for metrics in own_metrics_per_question])
    baseline_avg_recall_at_n[i] = np.average([metrics[i]['recall'] for metrics in baseline_metrics_per_question])
    
    own_mrr_at_n[i] = np.nanmean([metrics[i]['reciprocal_rank'] for metrics in own_metrics_per_question])
    baseline_mrr_at_n[i] = np.nanmean([metrics[i]['reciprocal_rank'] for metrics in baseline_metrics_per_question])
    
    own_mar_at_n[i] = np.nanmean([metrics[i]['average_rank'] for metrics in own_metrics_per_question])
    baseline_mar_at_n[i] = np.nanmean([metrics[i]['average_rank'] for metrics in baseline_metrics_per_question])

    print("-"*50)
    print(f"METRICS AT N={i:<2} | OWN RESULTS | BASELINE RESULTS")
    print("-"*50)
    print(f"RECALL          | {round(own_avg_recall_at_n[i],5):<11} | {round(baseline_avg_recall_at_n[i],5):<10}")
    print(f"MRR             | {round(own_mrr_at_n[i],5):<11} | {round(baseline_mrr_at_n[i],5):<10}")
    print(f"MAR             | {round(own_mar_at_n[i],5):<11} | {round(baseline_mar_at_n[i],5):<10}")



--------------------------------------------------
METRICS AT N=5  | OWN RESULTS | BASELINE RESULTS
--------------------------------------------------
RECALL          | 0.718       | 0.73733   
MRR             | 0.81208     | 0.82897   
MAR             | 2.11463     | 2.09299   
--------------------------------------------------
METRICS AT N=10 | OWN RESULTS | BASELINE RESULTS
--------------------------------------------------
RECALL          | 0.85467     | 0.865     
MRR             | 0.77884     | 0.79792   
MAR             | 2.97372     | 2.92247   
--------------------------------------------------
METRICS AT N=15 | OWN RESULTS | BASELINE RESULTS
--------------------------------------------------
RECALL          | 0.918       | 0.93333   
MRR             | 0.77024     | 0.78635   
MAR             | 3.61544     | 3.65995   
--------------------------------------------------
METRICS AT N=20 | OWN RESULTS | BASELINE RESULTS
--------------------------------------------------
RECALL   

In [9]:
import matplotlib.pyplot as plt
from pydantic import BaseModel, ValidationError, Field, model_validator 
from typing import List, Self
import math


class BarChartData(BaseModel):
    """
    Schema for validating bar chart data.
    Ensures data consistency before visualization.
    """
    labels: List[str] = Field(..., description="Names of the bars")
    values: List[float] = Field(..., description="Numerical values for the bars")
    title: str = Field(..., description="Title of the chart")

    @model_validator(mode="after")
    def check_length_match(self) -> Self:
        if len(self.labels) != len(self.values):
            raise ValidationError(f"There is different number of labels and values!\n{self.labels=}\n{self.values=}")
        return self
    

def create_annotated_bar_chart(data: BarChartData, output_file: str = 'annotated_bar_chart.png'):
    """
    Generates a bar chart with explicit value annotations above each bar.
    """
    # Create the figure and axis explicitly for better control
    fig, ax = plt.subplots()
    
    # Capture the container of bars to access their properties later
    bars = ax.bar(data.labels, data.values, color=['#1f77b4', '#ff7f0e'])
    
    ax.bar_label(bars, padding=3)

    ax.set_xlabel('Categories')
    ax.set_ylabel('Energy consumption (kWh)')
    ax.set_title(data.title)
    
    # Dynamic Y-Axis Adjustment
    # Crucial: Increase the y-axis limit by 10% to prevent the text from being cut off at the top
    ax.set_ylim(0, max(data.values) * 1.1)
    
    plt.savefig(output_file, dpi=300)
    plt.close() # Always close the plot to free memory in batch processing
    
def create_multi_bar_chart(data_list: List[BarChartData], output_file: str = 'multi_plot.png'):
    """
    Generates a figure containing subplots for each BarChartData instance.
    Dynamically calculates grid dimensions.
    """
    n = len(data_list)
    if n == 0:
        raise ValueError("Input list is empty.")

    # 1. Calculate Grid Dimensions
    # We aim for a roughly square grid, prioritizing width (max 3 columns)
    cols = min(n, 2)
    rows = math.ceil(n / cols)
    
    # Calculate figure size: 6 inches per col, 5 inches per row
    figsize = (6 * cols, 5 * rows)

    # 2. Create Subplots
    # squeeze=False ensures axes is always a 2D array, simplifying indexing
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    
    # Flatten axes array for easy 1D iteration
    axes_flat = axes.flatten()

    # 3. Plotting Loop
    for i, data in enumerate(data_list):
        ax = axes_flat[i]
        
        # Original Plotting Logic applied to specific axis
        bars = ax.bar(data.labels, data.values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
        
        # Native matplotlib annotation (requires matplotlib >= 3.4.0)
        ax.bar_label(bars, padding=3, fmt='%.5f')

        ax.set_xlabel('Categories')
        ax.set_ylabel('Values')
        ax.set_title(data.title)
        
        # Dynamic Y-Axis Adjustment for headroom
        if data.values:
            ax.set_ylim(0, max(data.values) * 1.15)

    # 4. Cleanup Unused Axes
    # If we have a 2x2 grid (4 slots) but only 3 data items, turn off the 4th slot
    for j in range(n, len(axes_flat)):
        axes_flat[j].axis('off')

    # Adjust layout to prevent overlapping titles/labels
    plt.tight_layout()
    
    plt.savefig(output_file, dpi=300)
    plt.close()


In [11]:
# --- Execution Example ---
# try:
#     dataset = []
#     for i in range(5,21,5):
#         dataset.append(BarChartData(labels=["Classical Hybrid Retrieval Baseline", "Hybrid Retrieval Document Slices"], values=[baseline_avg_recall_at_n[i], own_avg_recall_at_n[i]], title=f"Recall@{i}"))
    
#     create_multi_bar_chart(dataset, 'dashboard_view_recall_vs_hybrid.png')
#     print("Multi-plot generated successfully.")
# except Exception as e:
#     print(f"Error: {e}")

import csv

def extract_emissions(filepath: str) -> list:
    emissions_list = []
    energy_list = []
    
    with open(filepath, mode='r', encoding='utf-8') as file:
        reader = csv.DictReader(file)
        
        for row in reader:
            # CodeCarbon outputs values as strings; we cast them to floats
            emissions_list.append(float(row['emissions']))
            energy_list.append(float(row['energy_consumed']))
            
    return emissions_list, energy_list

my_emissions, energy_consumption = extract_emissions('emissions_data/emissions.csv')
my_emissions = [round(emission*1000,5) for emission in my_emissions]

emission_data = BarChartData(labels=["Anthropic Baseline", "Document slices"], values=[my_emissions[1], my_emissions[0]], title="CO2 equivalent emissions (g)")
energy_data = BarChartData(labels=["Anthropic Baseline", "Document slices"], values=[energy_consumption[1], energy_consumption[0]], title="Total energy consumed (kWh)")

# create_annotated_bar_chart(data=emission_data, output_file="benchmarking_results/preprocessing/emissions.png")
create_annotated_bar_chart(data=energy_data, output_file="benchmarking_results/preprocessing/energy_consumption.png")
create_multi_bar_chart([emission_data, energy_data], 'benchmarking_results/preprocessing/eco_friendliness.png')

# LLM as a judge (TODO)

In [9]:
# for question in tqdm(qna[0], desc="Processing questions..."):
	# question_prompt = qna[0]["question"]

	# df = table.search(question_prompt, query_type="hybrid", vector_column_name="vector", fts_columns="text") \
	#             .rerank(reranker=reranker) \
	#             .limit(10) \
	#             .to_pandas()

	# for idx, row in tqdm(df.iterrows(), total=len(df), desc="Grading Chunks"):
	#     resp = grade_chunk_relevance(question_prompt, row['text'], MODEL_TAG)
	#     df.at[idx, 'llm_grade'] = resp.score
	#     df.at[idx, 'reasoning'] = resp.chain_of_thought

	# df[['text', 'llm_grade', 'reasoning']]
		